In [1]:
import pandas as pd

# Define a raw data dataframe
raw_df = pd.read_csv("C:/Projetos/biofuels-sciml/data/raw/dataset.csv")
raw_df

,substance_1,substance_2,x_1,x_2,T,gamma_1,gamma_2
0,water,ethanol,0.0,1.0,305,2.69012,1.00000
1,water,ethanol,0.2,0.8,305,1.98665,1.03042
2,water,ethanol,0.4,0.6,305,1.64071,1.11962
3,water,ethanol,0.6,0.4,305,1.35956,1.35896
4,water,ethanol,0.8,0.2,305,1.12435,2.16471
...,...,...,...,...,...,...,...
139,ethanol,hexane,0.2,0.8,315,4.03860,1.16467
140,ethanol,hexane,0.4,0.6,315,4.03761,1.16474
141,ethanol,hexane,0.6,0.4,315,1.33527,2.29596
142,ethanol,hexane,0.8,0.2,315,1.07669,3.79283


In [2]:
from ugropy import Groups


# Define functions to calculate r and q from UNIFAC
def calculate_r(substance: str) -> float:
    # Load unifac parameters
    unifac_parameters = pd.read_csv(
        "C:/Projetos/biofuels-sciml/data/unifac_parameters/unifac_r_and_q.csv"
    )
    unifac_r_values = unifac_parameters.set_index("Subgroup")["R (volume)"].to_dict()

    init_substance = Groups(substance)
    substance_group = init_substance.unifac.subgroups

    # Calculate r value
    initial_r = 0
    for key in substance_group.keys():
        initial_r += substance_group[key] * unifac_r_values[key]

    return round(initial_r, 4)


def calculate_q(substance: str) -> float:
    # Load unifac parameters
    unifac_parameters = pd.read_csv(
        "C:/Projetos/biofuels-sciml/data/unifac_parameters/unifac_r_and_q.csv"
    )
    unifac_q_values = unifac_parameters.set_index("Subgroup")["Q (area)"].to_dict()

    init_substance = Groups(substance)
    substance_group = init_substance.unifac.subgroups

    # Calculate r value
    initial_q = 0
    for key in substance_group.keys():
        initial_q += substance_group[key] * unifac_q_values[key]

    return round(initial_q, 4)

In [3]:
# Create new dataset
substance_columns = [col for col in raw_df.columns if col.startswith("substance")]
num_substances = len(substance_columns)

for index, col in enumerate(substance_columns, start=1):
    raw_df[f"r_{index}"] = raw_df[col].apply(calculate_r)
    raw_df[f"q_{index}"] = raw_df[col].apply(calculate_q)

# Drop string columns
raw_df.drop(substance_columns, axis=1, inplace=True)

# Reorder the columns
new_order = [
    col
    for i in range(1, len(substance_columns) + 1)
    for col in [f"r_{i}", f"q_{i}", f"x_{i}"]
]
new_order += ["T"]
new_order += [
    col for i in range(1, len(substance_columns) + 1) for col in [f"gamma_{i}"]
]

processed_df = raw_df[new_order]

# Save table
processed_df.to_csv(
    "C:/Projetos/biofuels-sciml/data/processed/input_dataset.csv", index=False
)